# Lab 3.1 – Muster finden: drei Fragen an die Nordwind-Daten

**Starter.** Kopie in `project/playground/` bearbeiten. Zellen der Reihe nach mit **Umschalt+Enter** ausführen.

| Frage | Werkzeug | Ergebnis |
| --- | --- | --- |
| 1 · Wie wächst jede Region? | `pivot_table` + `pct_change` | `wachstum` |
| 2 · Welche Produktgruppe verdient am meisten je Euro Umsatz? | `groupby` + Summen | `marge` |
| 3 · Wie entwickeln sich die Kanäle Monat für Monat? | `pivot_table` | `kanal_monat` (Aufgabe 1) |

Eingabe: `umsatz_features.csv` aus dem Beispiel von Lab 2.3. Die drei Tabellen gehen als CSV nach `project/playground/`.

In [ ]:
import pandas as pd

EINGABE = "project/examples/m02-data-preparation/l03-features/umsatz_features.csv"
ORDNER = "project/playground/"     # hier landen die drei Ergebnistabellen

df = pd.read_csv(EINGABE, parse_dates=["datum"])   # Spalte datum gleich als Datum lesen
print(len(df), "Zeilen von", df["monat"].min(), "bis", df["monat"].max())
df.head(3)

## Überblick: Wie sind die Werte verteilt?

`describe()` zeigt je Zahlenspalte Anzahl, Mittelwert (`mean`), Streuung (`std`), Minimum, Viertelwerte und Maximum.

In [ ]:
df[["menge", "umsatz", "marge"]].describe().round(2)

## Frage 1: Wie wächst jede Region?

Verglichen werden je Jahr nur die Monate Januar bis September, weil 2026 erst bis September vorliegt.
`pct_change(axis="columns")` rechnet die Veränderung von Spalte zu Spalte, also Jahr gegen Vorjahr.

In [ ]:
def wachstum_je_region(df, bis_monat=9):
    """Frage 1: Umsatz Januar bis bis_monat je Region und Jahr, dazu Veränderung zum Vorjahr in %."""
    zeitraum = df[df["datum"].dt.month <= bis_monat]              # gleiche Monate in jedem Jahr
    tabelle = zeitraum.pivot_table(index="region", columns="jahr", values="umsatz", aggfunc="sum")
    letztes_jahr = tabelle.columns.max()
    veraenderung = tabelle.pct_change(axis="columns") * 100       # Jahr gegen Vorjahr in %
    tabelle["wachstum_prozent"] = veraenderung[letztes_jahr]
    tabelle.columns = [str(spalte) for spalte in tabelle.columns]  # 2024 -> "2024"
    return tabelle.round(1).sort_values("wachstum_prozent", ascending=False)


wachstum = wachstum_je_region(df)
wachstum

## Frage 2: Welche Produktgruppe verdient am meisten je Euro Umsatz?

Marge einer Gruppe = Summe Deckungsbeitrag / Summe Umsatz (Regel aus Lab 2.3), nie der Mittelwert der Zeilenmargen.

In [ ]:
def marge_je_produktgruppe(df):
    """Frage 2: Marge je Produktgruppe = Summe Deckungsbeitrag / Summe Umsatz."""
    summen = df.groupby("produktgruppe")[["umsatz", "deckungsbeitrag"]].sum()
    summen["marge_prozent"] = summen["deckungsbeitrag"] / summen["umsatz"] * 100
    return summen.round(1).sort_values("marge_prozent", ascending=False)


marge = marge_je_produktgruppe(df)
marge

## Frage 3: Wie entwickeln sich die Kanäle Monat für Monat?

Gesucht ist eine Tabelle `kanal_monat`: eine Zeile je Monat, je eine Spalte Umsatz für `Außendienst` und `Onlineshop`, dazu `anteil_online_prozent`.
Die Zelle darunter mit dem vorgegebenen Prompt aus der Übung (Aufgabe 3) füllen, dann die Prüfzelle ausführen.

In [ ]:
# >>> AUFGABE 1: Tabelle kanal_monat erzeugen – Prompt siehe Übung, Aufgabe 3
kanal_monat = None

# <<< ENDE AUFGABE 1
kanal_monat

In [ ]:
# Prüfzelle – nicht ändern
def pruefe_kanal_monat(t):
    if not isinstance(t, pd.DataFrame):
        return "nicht OK – kanal_monat ist noch keine Tabelle (DataFrame)."
    fehlt = {"Außendienst", "Onlineshop", "anteil_online_prozent"} - set(t.columns)
    if fehlt:
        return "nicht OK – Spalten fehlen: " + ", ".join(sorted(fehlt)) + ". Kanäle als Spalten: pivot_table mit columns=\"kanal\"."
    if len(t) != 33:
        return f"nicht OK – {len(t)} Zeilen statt 33 (eine Zeile je Monat, 2024-01 bis 2026-09)."
    anteil = float(t["anteil_online_prozent"].iloc[-1])
    if abs(anteil - 30.1) > 0.1:
        return f"nicht OK – Onlineshop-Anteil im letzten Monat {anteil:.1f} statt 30.1: Onlineshop / (Außendienst + Onlineshop) * 100."
    return "OK – 33 Monate, Onlineshop-Anteil im September 2026: 30,1 %"


print(pruefe_kanal_monat(kanal_monat))

## Ergebnisse speichern

Die drei Tabellen gehen als CSV in den Ordner `ORDNER`. Lab 3.2 liest dieselben Dateien aus dem Beispielordner.

In [ ]:
wachstum.to_csv(ORDNER + "wachstum_region.csv")
marge.to_csv(ORDNER + "marge_produktgruppe.csv")
if isinstance(kanal_monat, pd.DataFrame):
    kanal_monat.to_csv(ORDNER + "kanal_monat.csv")
    print("Gespeichert: drei Tabellen in", ORDNER)
else:
    print("Gespeichert: zwei Tabellen in", ORDNER, "– kanal_monat fehlt noch (Aufgabe 1)")